In [ ]:
import re
from collections import Counter
import csv
from pathlib import Path
from torchvision import transforms

PAD, UNK, BOS, EOS = 0, 1, 2, 3
MAX_TEXT_LENGTH = 40

class TokenizeCaption:
    def __init__(self):
        self.vocab = None
    
    def tokenize(self, text):
        return re.findall(r"\w+|[^\w\s]", text.lower())

    def build_vocab(self, metadata_path):
        counts = Counter()
        with open(metadata_path, newline="", encoding="utf-8") as metadata_file:
            reader = csv.DictReader(metadata_file)
            if reader.fieldnames is None or "caption" not in reader.fieldnames:
                raise ValueError(f"{metadata_path} must contain a 'caption' column")

            for row in reader:
                counts.update(self.tokenize(row["caption"]))

        vocab = {"<pad>": PAD, "<unk>": UNK, "<bos>": BOS, "<eos>": EOS}
        for token in counts:
            if token not in vocab:
                vocab[token] = len(vocab)
        self.vocab = vocab
        # save vocab to a file
        vocab_path = Path(__file__).resolve().parent / "data" / "SHAPE" / "vocab.json"
        import json
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(self.vocab, f)

    def encode_caption(self, text):
        if self.vocab is None:
            raise ValueError("Vocabulary has not been built or loaded.")
        
        tokens = self.tokenize(text)[:MAX_TEXT_LENGTH - 2]
        ids = [BOS, *(self.vocab.get(token, UNK) for token in tokens), EOS]
        ids.extend([PAD] * (MAX_TEXT_LENGTH - len(ids)))
        return ids

    def load_vocab(self):
        vocab_path = Path(__file__).resolve().parent / "data" / "SHAPE" / "vocab.json"
        import json
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)
        print("Loaded vocabulary:", self.vocab)
        return self.vocab

class image_transform:
    def __init__(self, image_resolution):
        self.transform = transforms.Compose([
            transforms.Resize((image_resolution, image_resolution)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def __call__(self, image):
        return self.transform(image)
